Premiers essais du projet !

In [33]:
import tensorflow as tf
#import tensorflow_decision_forests as tfdf
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [34]:
dataset_df = pd.read_csv('train.csv')
print("Full train dataset shape is {}".format(dataset_df.shape))

Full train dataset shape is (8693, 14)


In [ ]:
dataset_df.describe()

In [ ]:
dataset_df.info()

In [ ]:
plot_df = dataset_df.Transported.value_counts()
plot_df.plot(kind="bar")

In [ ]:
fig, ax = plt.subplots(5,1,  figsize=(10, 10))
plt.subplots_adjust(top = 2)

sns.histplot(dataset_df['Age'], color='b', bins=50, ax=ax[0]);
sns.histplot(dataset_df['FoodCourt'], color='b', bins=50, ax=ax[1]);
sns.histplot(dataset_df['ShoppingMall'], color='b', bins=50, ax=ax[2]);
sns.histplot(dataset_df['Spa'], color='b', bins=50, ax=ax[3]);
sns.histplot(dataset_df['VRDeck'], color='b', bins=50, ax=ax[4]);

In [ ]:
valeurs_uniques = dataset_df['HomePlanet'].unique()
print(valeurs_uniques)


<h2>Modification de la colonne Cabin</h2>

In [ ]:
dataset_df[['Deck', 'Num', 'Side']] = dataset_df['Cabin'].str.split('/', expand=True)

#traitement de la colone Side
side_mapping={"P":1,"S":-1}
dataset_df['Side'] = dataset_df['Side'].map(side_mapping)

#traitement de la colone Deck
deck_mapping={"A":1,"B":2,"C":3,"D":4,"E":5,"F":6,"G":7}
dataset_df['Deck'] = dataset_df['Deck'].map(deck_mapping)


In [ ]:
fig, ax = plt.subplots(3,1,  figsize=(10, 10))
plt.subplots_adjust(top = 2)
sns.histplot(dataset_df['Deck'], color='b', bins=50, ax=ax[0]);
sns.histplot(dataset_df['Num'], color='b', bins=50, ax=ax[1]);
sns.histplot(dataset_df['Side'], color='b', bins=50, ax=ax[2]);

In [ ]:
transported_df = dataset_df[dataset_df['Transported'] == True]
sns.countplot(x='Deck', data=transported_df)
plt.title('Nombre de personnes transportées par Deck')
plt.xlabel('Deck')
plt.ylabel('Nombre de personnes')
plt.show()

<h3>Dans le prochain graphe, on a la proportion de personne transporté au sein d'un meme deck, cela permet d'identifier la propbabilité statistique d'être transporté pour chaque deck.</h3>

In [ ]:
proportions = dataset_df.groupby('Deck')['Transported'].mean()
proportions.plot(kind='bar')
plt.title('Proportion de personnes transportées par Deck')
plt.xlabel('Deck')
plt.ylabel('Proportion de personnes transportées')
plt.show()

<h3>Dans le graphe suivant, on affiche la proportion de personnes transportées dans chaque deck par rapport au nombre total de personne transportées. Cela nous informe d'où viennent la majorité des passagers transportés mais ne nous informe pas sur la proportion de transpotées au sein d'un meme deck.</h3>

In [ ]:
total_transported = dataset_df['Transported'].sum()
proportions = dataset_df[dataset_df['Transported'] == True].groupby('Deck').size() / total_transported
proportions.plot(kind='bar')
plt.title('Proportion de personnes transportées par Deck')
plt.xlabel('Deck')
plt.ylabel('Proportion de personnes transportées')
plt.show()

<h2>Traitement de PassengerId</h2>

In [ ]:
dataset_df[['group','num_in_the_group']] = dataset_df['PassengerId'].str.split('_', expand=True)
dataset_df=dataset_df.drop(columns=['num_in_the_groupe'])
dataset_df.head()

In [ ]:
transported_df = dataset_df[dataset_df['Transported'] == True]
sns.countplot(x='group', data=transported_df)
plt.title('Nombre de personnes transportées par groupe')
plt.xlabel('groupe')
plt.ylabel('Nombre de personnes')
plt.show()

<h3>Aucune tendance ne semble se dégager concernant le numéro du groupe de passagers. On poura cependant par la suite essayer de trouver un lien entre le groupe et d'autre variable telle que le nom de famille ou le numéro de cabine, ou bien encore le deck.</h3>

In [ ]:
proportions = dataset_df.groupby('num_in_the_group')['Transported'].mean()
proportions.plot(kind='bar')
plt.title('Proportion de personnes transportées par numéro au sein du groupe')
plt.xlabel('Numéro au sein du groupe')
plt.ylabel('Proportion de personnes transportées')
plt.show()

<h3>Le numéro au sein du groupe semble avoir un impact sur la probabilité d'être transporté.</h3>

<h1><i>Fonction de traitement des données</i></h1>

In [35]:
def traitement_dataset(dataset_df):
    dataset_df[['Deck', 'Num', 'Side']] = dataset_df['Cabin'].str.split('/', expand=True)
    side_mapping={"P":1,"S":-1}
    dataset_df['Side'] = dataset_df['Side'].map(side_mapping)

    deck_dummies = pd.get_dummies(dataset_df['Deck'], prefix='')
    dataset_df = pd.concat([dataset_df, deck_dummies], axis=1)

    #dataset_df[['group','num_in_the_group']] = dataset_df['PassengerId'].str.split('_', expand=True) ici c'est inutile car passengerID n'apporte aucune info utile

    # Mapping by Mael
    planet_mapping = {"Earth": 1, "Europa": 2, "Mars": 3} #il faudra peut êrte changer les valeurs car à priori on ne sait pas si Earth est plus proche que Mars Europa ou Mars on un odre d'importance
    cryo_sleep_mapping = {False: 0, True: 1}
    destination_mapping = {"TRAPPIST-1e": 1, "55 Cancri e": 2, "PSO J318.5-22": 3} #idem
    vip_mapping = {False: 0, True: 1}
    transported_mapping = {False: 0, True: 1}
    # Modify data according to mappings by Mael
    dataset_df["HomePlanet"] = dataset_df["HomePlanet"].map(planet_mapping)
    dataset_df["CryoSleep"] = dataset_df["CryoSleep"].map(cryo_sleep_mapping)
    dataset_df["Destination"] = dataset_df["Destination"].map(destination_mapping)
    dataset_df["VIP"] = dataset_df["VIP"].map(vip_mapping)
    dataset_df["Transported"] = dataset_df["Transported"].map(transported_mapping)

    # Drop columns
    dataset_df.drop(columns=['PassengerId', 'Name', 'Cabin', 'Deck'],inplace=True)
    

In [36]:
traitement_dataset(dataset_df)
dataset_df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported,Deck,Num,Side
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False,B,0,1.0
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True,F,0,-1.0
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False,A,0,-1.0
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False,A,0,-1.0
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True,F,1,-1.0


<h1>
<i>Traitement des valeurs manquantes</i>
</h1>

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,...,Num,Side,_A,_B,_C,_D,_E,_F,_G,_T
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,...,0,1.0,False,True,False,False,False,False,False,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,...,0,-1.0,False,False,False,False,False,True,False,False
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,...,0,-1.0,True,False,False,False,False,False,False,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,...,0,-1.0,True,False,False,False,False,False,False,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,...,1,-1.0,False,False,False,False,False,True,False,False
